# Phase 3E.03 — Transaction Costs, Turnover & Net Performance

## 1. Research Objective

This notebook evaluates the economic impact of portfolio turnover
and transaction costs on the 12-month TSMOM strategy.

The analysis compares gross and net performance against the
Equal-Weighted Long-Only benchmark across Train and Validation.

The Test period remains untouched until Phase 3F.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if not (project_root / "src").is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
import src.data
import src.metrics
import src.research
import src.signals
import src.strategy

from matplotlib.ticker import PercentFormatter
from src.data import prepare_monthly_prices
from src.strategy import build_tsmom_positions, calculate_portfolio_weights, calculate_portfolio_exposure, calculate_gross_portfolio_returns
from src.metrics import calculate_performance_metrics

Matplotlib is building the font cache; this may take a moment.


## 2. Data & Portfolio Setup

Reconstruct the 12-month TSMOM portfolio and its Equal-Weighted Long-Only benchmark using the existing research infrastructure.

$$
p_{i,t}=s_{i,t-1},\qquad w_{i,t}=\frac{p_{i,t}}{N},\quad N=16
$$

$$
r_{i,t}=\frac{P_{i,t}}{P_{i,t-1}}-1
$$

$$
R_{p,t}^{gross}=\sum_{i=1}^{N}w_{i,t}r_{i,t}
$$

$$
w_{i,t}^{B}=\frac{1}{N},\qquad R_{B,t}=\frac{1}{N}\sum_{i=1}^{N}r_{i,t}
$$

**Notation and implementation.** $i$ denotes an ETF, $t$ the holding month, $N=16$, $s_{i,t-1}$ the preceding month-end signal, and $P_{i,t}$ the adjusted month-end price. $p_{i,t}$ and $w_{i,t}$ correspond to `positions` and `portfolio_weights`; $r_{i,t}$ is `asset_returns`; $R_{p,t}^{gross}$ is `gross_portfolio_returns`. The benchmark uses $w_{i,t}^{B}$ (`benchmark_weights`) and $R_{B,t}$ (`benchmark_returns`).

In [2]:
asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

start_date = "2008-01-01"
end_date = "2022-01-01"
lookback = 12

monthly_prices = prepare_monthly_prices(tickers, start_date, end_date)

signals, positions = build_tsmom_positions(monthly_prices, lookback=lookback)

portfolio_weights = calculate_portfolio_weights(positions)

gross_exposure, net_exposure = calculate_portfolio_exposure(portfolio_weights)

print(monthly_prices.head())
print(monthly_prices.shape)

print(positions.head())
print(positions.shape)

print(portfolio_weights.dropna().head())
print(portfolio_weights.shape)

Ticker            DBA        DBB        DBC        EEM        EWJ        FXA  \
Date                                                                           
2008-01-31  31.200171  19.611864  27.033497  30.937822  36.718548  65.858337   
2008-02-29  35.054859  21.755016  30.046469  31.554819  36.170078  68.784767   
2008-03-31  30.744696  20.388252  29.771803  30.370548  35.708221  67.721024   
2008-04-30  30.871216  20.388252  31.569599  33.145905  38.335094  70.257133   
2008-05-31  29.816872  19.385418  33.750252  34.194565  39.085621  71.677879   

Ticker             FXB         FXE         FXY        HYG        IEF  \
Date                                                                   
2008-01-31  173.245667  136.197556   93.910004  30.854895  56.444126   
2008-02-29  173.586060  139.362045   96.129997  30.278103  57.143585   
2008-03-31  174.066635  145.244675  100.239998  30.564568  57.907795   
2008-04-30  175.017456  144.052902   95.949997  31.602829  56.507965   
2008-05

In [3]:
asset_returns, weighted_returns, gross_portfolio_returns = calculate_gross_portfolio_returns(monthly_prices, portfolio_weights)

print(asset_returns.head())
print(asset_returns.shape)

print(weighted_returns.head())
print(weighted_returns.shape)

print(gross_portfolio_returns.dropna().head())
print(gross_portfolio_returns.shape)

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2008-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-02-29  0.123547  0.109278  0.111453  0.019943 -0.014937  0.044435   
2008-03-31 -0.122955 -0.062825 -0.009141 -0.037531 -0.012769 -0.015465   
2008-04-30  0.004115  0.000000  0.060386  0.091383  0.073565  0.037449   
2008-05-31 -0.034153 -0.049187  0.069074  0.031638  0.019578  0.020222   

Ticker           FXB       FXE       FXY       HYG       IEF       LQD  \
Date                                                                     
2008-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2008-02-29  0.001965  0.023235  0.023640 -0.018694  0.012392 -0.007063   
2008-03-31  0.002769  0.042211  0.042755  0.009461  0.013374 -0.005971   
2008-04-30  0.005462 -0.008205 -0.042797  0.033969 -0.024173  0.012446   
2008-05-31  0.000468 -0.000179 -0.013

In [4]:
benchmark_weights = pd.DataFrame(1/len(portfolio_weights.columns), index=portfolio_weights.index, columns=portfolio_weights.columns)

print(benchmark_weights.head())

benchmark_returns = (asset_returns * benchmark_weights).sum(axis=1, min_count=16)

print(benchmark_returns.head())
print(benchmark_returns.shape)

Ticker         DBA     DBB     DBC     EEM     EWJ     FXA     FXB     FXE  \
Date                                                                         
2008-01-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   
2008-02-29  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   
2008-03-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   
2008-04-30  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   
2008-05-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625   

Ticker         FXY     HYG     IEF     LQD     SPY     TLT     USO     VGK  
Date                                                                        
2008-01-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  
2008-02-29  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  
2008-03-31  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  
2008-04-30  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  0.0625  

## 3. Portfolio Turnover

### 3.1 Signal-Driven Turnover

Signal-driven turnover measures changes in target portfolio weights, excluding weight drift caused by asset returns.

$$
TO_t^{signal}=\sum_{i=1}^{N}|w_{i,t}-w_{i,t-1}|
$$

For $n$ valid monthly observations, annualized average turnover is $12\left(\sum_t TO_t^{signal}/n\right)$. The convention measures total traded notional without dividing by two.

**Notation and implementation.** $w_{i,t}$ and $w_{i,t-1}$ are consecutive signed target weights. $TO_t^{signal}$ is the total absolute target-weight change for month $t$, stored in `signal_turnover`. `weight_changes` stores signed differences before the absolute value. The annualized mean is computed using `signal_turnover.loc["2010":"2021"].mean() * 12`.

In [5]:
weight_changes = portfolio_weights.diff()

signal_turnover = weight_changes.abs().sum(axis=1, min_count=16)

print(signal_turnover.loc["2010":"2021"].mean())
print(signal_turnover.loc["2010":"2021"].max())
print(signal_turnover.loc["2010":"2021"].median())
print(signal_turnover.loc["2010":"2021"].mean() * 12)

0.21961805555555555
1.0
0.1875
2.6354166666666665


**Results (2010–2021):** average monthly signal-driven turnover 21.96%, median 18.75%, maximum 100.00%, annualized average 263.54%. These estimates exclude weight drift.

### 3.2 Weight Drift

Asset returns cause portfolio weights to deviate from their initial target allocation before rebalancing. Under the gross-return model, assuming no additional trades during the holding month:

$$
w_{i,t}^{drift}=w_{i,t}\frac{1+r_{i,t}}{1+R_{p,t}^{gross}}
$$

The expression applies to signed long and short weights, provided portfolio wealth remains positive. It excludes financing, borrow fees and transaction costs.

**Notation and implementation.** $w_{i,t}$ is the beginning-of-month signed target weight (`portfolio_weights`), $r_{i,t}$ is the ETF return (`asset_returns`), and $R_{p,t}^{gross}$ is the total gross portfolio return (`gross_portfolio_returns`). The numerator marks the position to market; the denominator scales it by current portfolio wealth. $w_{i,t}^{drift}$ is the end-of-month pre-rebalancing weight stored in `drifted_weights`.

In [6]:
drifted_weights = (portfolio_weights * (1 + asset_returns)).div(
    1 + gross_portfolio_returns, axis=0
)

print(drifted_weights.dropna().head())
print(drifted_weights.shape)

Ticker           DBA       DBB       DBC       EEM       EWJ       FXA  \
Date                                                                     
2009-02-28 -0.057034 -0.062531 -0.056831 -0.056446 -0.052649 -0.060677   
2009-03-31 -0.068198 -0.074534 -0.068839 -0.076867 -0.070506 -0.071663   
2009-04-30 -0.068243 -0.069642 -0.066018 -0.076366 -0.071187 -0.069207   
2009-05-31 -0.075425 -0.071199 -0.079519 -0.079296 -0.075306 -0.075460   
2009-06-30 -0.056914 -0.064956 -0.060733 -0.060967 -0.063110 -0.062875   

Ticker           FXB       FXE       FXY       HYG       IEF       LQD  \
Date                                                                     
2009-02-28 -0.059518 -0.059610  0.055399 -0.054370  0.059778 -0.057087   
2009-03-31 -0.065952 -0.068956  0.064808 -0.067030  0.067955 -0.066086   
2009-04-30 -0.068064 -0.065838  0.066328 -0.075234  0.064270 -0.067919   
2009-05-31 -0.074736 -0.073083  0.070767 -0.070347  0.066973 -0.069979   
2009-06-30 -0.063471 -0.061912  0.061

### 3.3 Actual Portfolio Turnover — Pending Implementation

Actual turnover compares the current target weights with the previous month's post-return weights:

$$
TO_t^{actual}=\sum_{i=1}^{N}|w_{i,t}-w_{i,t-1}^{drift}|
$$

This is the planned gross-model turnover convention, not a completed calculation. Initial entry handling, benchmark turnover, transaction costs and net performance remain to be implemented. Test remains excluded.

**Planned output (not yet coded).** $w_{i,t}$ is the new target allocation for holding month $t$, and $w_{i,t-1}^{drift}$ is the prior month-end pre-trade allocation from `drifted_weights.shift(1)`. $TO_t^{actual}$ will measure total absolute traded notional as a fraction of portfolio wealth. No actual-turnover output variable or result exists yet.

In [ ]:
weight_differences = portfolio_weights - drifted_weights.shift(1)

actual_turnover = weight_differences.abs().sum(axis=1, min_count=16)

actual_turnover_research = actual_turnover.loc["2010":"2021"]
signal_turnover_research = signal_turnover.loc["2010":"2021"]

mean_turnover = actual_turnover_research.mean()
median_turnover = actual_turnover_research.median()
max_turnover = actual_turnover_research.max()
annualized_turnover = mean_turnover * 12
mean_turnover_difference = (actual_turnover_research - signal_turnover_research).mean()

print(f"Mean turnover : {mean_turnover:.2%}")
print(f"Median turnvoer : {median_turnover:.2%}")
print(f"Max turnover : {max_turnover:.2%}")
print(f"Annualized turnover : {annualized_turnover:.2%}")
print(f"Mean turnover difference : {mean_turnover_difference:.2%}")

print(f"Months with actual turnover below signal turnover : {(actual_turnover_research < signal_turnover_research).sum()}")

Mean turnover : 24.80%
Median turnvoer : 23.99%
Max turnover : 102.16%
Annualized turnover : 297.62%
Mean turnover difference : 2.84%
Difference between real turnover & signal turnover : 1


### 3.4 Benchmark Turnover

The Equal-Weighted Long-Only benchmark maintains constant
target weights across all assets:

$$
w_{i,t}^{B} = \frac{1}{N}
$$

Despite constant target weights, asset returns generate
weight drift, requiring monthly rebalancing.

**Benchmark Weight Drift**

$$
\widetilde{w}_{i,t}^{B} =
w_{i,t}^{B}\frac{1+r_{i,t}}{1+R_{B,t}}
$$

**Benchmark Turnover**

$$
TO_t^{B} = \sum_{i=1}^{N}
\left|w_{i,t}^{B}-\widetilde{w}_{i,t-1}^{B}\right|
$$

**Notation and implementation**

| Symbol | Definition | Python variable |
|---|---|---|
| $N$ | Number of assets (16) | `len(tickers)` |
| $w_{i,t}^{B}$ | Benchmark target weight | `benchmark_weights` |
| $r_{i,t}$ | Monthly asset return | `asset_returns` |
| $R_{B,t}$ | Monthly benchmark return | `benchmark_returns` |
| $\widetilde{w}_{i,t}^{B}$ | Drifted benchmark weight | `benchmark_drifted_weights` |
| $TO_t^{B}$ | Actual benchmark turnover | `benchmark_turnover` |

Benchmark turnover is required to incorporate transaction
costs consistently across the strategy and its benchmark.

In [50]:
benchmark_drifted_weights = (benchmark_weights * (1 + asset_returns)).div(
    1 + benchmark_returns, axis=0)

print(np.allclose(benchmark_drifted_weights.sum(axis=1, min_count=16).dropna(), 1))

benchmark_turnover = benchmark_weights - benchmark_drifted_weights.shift(1)

benchmark_actual_turnover = benchmark_turnover.abs().sum(axis=1, min_count=16)

benchmark_actual_turnover_research = benchmark_actual_turnover.loc["2010":"2021"]

benchmark_mean_actual_turnover = benchmark_actual_turnover_research.mean()
benchmark_median_actual_turnover = benchmark_actual_turnover_research.median()
benchmark_max_actual_turnover = benchmark_actual_turnover_research.max()
benchmark_annualized_actual_turnover = benchmark_mean_actual_turnover * 12

print(f"Benchmark mean turnover : {benchmark_mean_actual_turnover:.2%}")
print(f"Benchmark median turnover : {benchmark_median_actual_turnover:.2%}")
print(f"Benchmark max turnover : {benchmark_max_actual_turnover:.2%}")
print(f"Benchmark annualized turnover : {benchmark_annualized_actual_turnover:.2%}")

True
Benchmark mean turnover : 2.57%
Benchmark median turnover : 2.24%
Benchmark max turnover : 9.54%
Benchmark annualized turnover : 30.83%


## 4. Transaction Cost Modeling

### 4.1 Proportional Transaction Costs

Transaction costs are modeled as a linear function
of actual portfolio turnover.

$$
TC_t^{S} = c \times TO_t^{actual}
$$

$$
TC_t^{B} = c \times TO_t^{B}
$$

**Notation and implementation**

| Symbol | Definition | Python variable |
|---|---|---|
| $c$ | Transaction cost rate | `cost_rate` |
| $TO_t^{actual}$ | Strategy turnover | `actual_turnover` |
| $TO_t^{B}$ | Benchmark turnover | `benchmark_actual_turnover` |
| $TC_t^{S}$ | Strategy transaction costs | `strategy_transaction_costs` |
| $TC_t^{B}$ | Benchmark transaction costs | `benchmark_transaction_costs` |

The initial transaction cost assumption is 10 basis points:

$$
c = \frac{10}{10\,000} = 0.001
$$

The same cost rate is applied to both portfolios.
Alternative cost assumptions will be evaluated during
the cost sensitivity analysis.

In [59]:
cost_rate = 10 / 10000

strategy_transaction_cost = cost_rate * actual_turnover

benchmark_transaction_cost = cost_rate * benchmark_actual_turnover

strategy_transaction_cost_research = strategy_transaction_cost.loc["2010":"2021"]
benchmark_transaction_cost_researhc = benchmark_transaction_cost.loc["2010":"2021"]

mean_strategy_transaction_cost = strategy_transaction_cost_research.mean()
median_strategy_transaction_cost = strategy_transaction_cost_research.median()
max_strategy_transaction_cost = strategy_transaction_cost_research.max()
annualized_strategy_transaction_cost = mean_strategy_transaction_cost * 12

mean_benchmark_transaction_cost = benchmark_transaction_cost_researhc.mean()
median_benchmark_transaction_cost = benchmark_transaction_cost_researhc.median()
max_benchmark_transaction_cost = benchmark_transaction_cost_researhc.max()
annualized_benchmark_transaction_cost = mean_benchmark_transaction_cost * 12

print(f"Mean strategy transaction cost : {mean_strategy_transaction_cost:.4%}")
print(f"Median strategy transaction cost : {median_strategy_transaction_cost:.4%}")
print(f"Max strategy transaction cost : {max_strategy_transaction_cost:.4%}")
print(f"Annualized strategy transaction cost : {annualized_strategy_transaction_cost:.4%}")

print("\n")

print(f"Mean benchmark transaction cost : {mean_benchmark_transaction_cost:.4%}")
print(f"Median benchmark transaction cost : {median_benchmark_transaction_cost:.4%}")
print(f"Max benchmark transaction cost : {max_benchmark_transaction_cost:.4%}")
print(f"Annualized benchmark transaction cost : {annualized_benchmark_transaction_cost:.4%}")

Mean strategy transaction cost : 0.0248%
Median strategy transaction cost : 0.0240%
Max strategy transaction cost : 0.1022%
Annualized strategy transaction cost : 0.2976%


Mean benchmark transaction cost : 0.0026%
Median benchmark transaction cost : 0.0022%
Max benchmark transaction cost : 0.0095%
Annualized benchmark transaction cost : 0.0308%


### 4.2 Net Portfolio Returns

Net portfolio returns are calculated by deducting
transaction costs from gross portfolio returns.

$$
R_{S,t}^{net} = R_{S,t}^{gross} - TC_t^S
$$

$$
R_{B,t}^{net} = R_{B,t}^{gross} - TC_t^B
$$

**Notation and implementation**

| Symbol | Definition | Python variable |
|---|---|---|
| $R_{S,t}^{gross}$ | Gross strategy return | `gross_portfolio_returns` |
| $R_{B,t}^{gross}$ | Gross benchmark return | `benchmark_returns` |
| $TC_t^S$ | Strategy transaction costs | `strategy_transaction_cost` |
| $TC_t^B$ | Benchmark transaction costs | `benchmark_transaction_cost` |
| $R_{S,t}^{net}$ | Net strategy return | `net_portfolio_returns` |
| $R_{B,t}^{net}$ | Net benchmark return | `net_benchmark_returns` |

This additive cost model expresses transaction costs
as a fraction of beginning-of-period portfolio wealth.

Financing costs and short-borrow fees are not included.

In [72]:
net_portfolio_returns = gross_portfolio_returns - strategy_transaction_cost

net_benchmark_returns = benchmark_returns - benchmark_transaction_cost

print((net_portfolio_returns <= gross_portfolio_returns).loc["2010":"2021"].all())
print((net_benchmark_returns <= benchmark_returns).loc["2010":"2021"].all())

True
True


## 5. Net Performance Analysis

### 5.1 Performance Dataset Preparation

Compare gross and net returns for the TSMOM strategy
and the Equal-Weighted Long-Only benchmark.

All portfolios must be evaluated over identical dates:

- Train: 2010–2017.
- Validation: 2018–2021.

The Test period remains excluded.

This analysis isolates the impact of modeled transaction
costs on portfolio performance.

In [100]:
performance_returns = pd.DataFrame({
    "Strategy Gross" : gross_portfolio_returns,
    "Strategy Net" : net_portfolio_returns,
    "Benchmark Gross" : benchmark_returns,
    "Benchmark Net" : net_benchmark_returns
})

display(performance_returns.loc["2010":"2021"])

train_returns = performance_returns.loc["2010":"2017"]
validation_returns = performance_returns.loc["2018":"2021"]

print(train_returns.shape)
print(train_returns.isna().sum())

print("\n")

print(validation_returns.shape)
print(validation_returns.isna().sum())

,Strategy Gross,Strategy Net,Benchmark Gross,Benchmark Net
Date,,,,
2010-01-31,-0.040241,-0.040632,-0.029985,-0.030016
2010-02-28,0.012361,0.012320,0.014269,0.014230
2010-03-31,0.020523,0.020254,0.017951,0.017931
2010-04-30,-0.003259,-0.003531,0.006469,0.006434
2010-05-31,-0.060898,-0.061044,-0.061139,-0.061158
...,...,...,...,...
2021-08-31,0.002943,0.002799,0.000926,0.000910
2021-09-30,0.004831,0.004693,-0.006289,-0.006303
2021-10-31,0.022365,0.022211,0.021550,0.021525


(96, 4)
Strategy Gross     0
Strategy Net       0
Benchmark Gross    0
Benchmark Net      0
dtype: int64


(48, 4)
Strategy Gross     0
Strategy Net       0
Benchmark Gross    0
Benchmark Net      0
dtype: int64


### 5.2 Gross vs. Net Performance Metrics

Calculate performance and risk metrics for gross and net
strategy and benchmark returns across Train and Validation.

The analysis relies on the reusable
`calculate_performance_metrics()` function defined in
`src/metrics.py`.

The objective is to quantify the impact of transaction
costs on absolute and risk-adjusted performance.

All comparisons use identical evaluation periods.

In [147]:
train_metrics = {}

for column in train_returns.columns:
    train_metrics[column] = calculate_performance_metrics(train_returns[column], risk_free_rate=0)

print(type(train_metrics))
print(type(train_metrics[column]))

train_metrics_df = pd.DataFrame(train_metrics).T
# display(train_metrics_df)

train_metrics_df_formatted = train_metrics_df.style.format({
    "CAGR" : "{:.2%}",
    "Annualized Volatility" : "{:.2%}",
    "Sharpe Ratio" : "{:.3f}",
    "Max Drawdown" : "{:.2%}"
})

display(train_metrics_df_formatted)

validation_metrics = {}

for column in validation_returns.columns:
    validation_metrics[column] = calculate_performance_metrics(validation_returns[column], risk_free_rate=0)

print(type(validation_metrics))
print(type(validation_metrics[column]))

validation_metrics_df = pd.DataFrame(validation_metrics).T
# display(validation_metrics_df)

validation_metrics_df_formatted = validation_metrics_df.style.format({
    "CAGR" : "{:.2%}",
    "Annualized Volatility" : "{:.2%}",
    "Sharpe Ratio" : "{:.3f}",
    "Max Drawdown" : "{:.2%}"
})

display(validation_metrics_df_formatted)

<class 'dict'>
<class 'dict'>


,CAGR,Annualized Volatility,Sharpe Ratio,Max Drawdown
Strategy Gross,0.74%,6.57%,0.145,-15.12%
Strategy Net,0.43%,6.57%,0.098,-15.37%
Benchmark Gross,1.96%,8.16%,0.278,-21.25%
Benchmark Net,1.93%,8.16%,0.275,-21.29%


<class 'dict'>
<class 'dict'>


,CAGR,Annualized Volatility,Sharpe Ratio,Max Drawdown
Strategy Gross,0.32%,6.59%,0.081,-9.31%
Strategy Net,0.04%,6.58%,0.039,-9.60%
Benchmark Gross,4.13%,9.60%,0.469,-14.75%
Benchmark Net,4.09%,9.60%,0.466,-14.76%
